# Lesson 09 Lab — Residual, Concat, and Dependency-Graph Pruning

**Puzzle:** Which tensors must change together when one residual branch loses channels?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

Structural pruning becomes a graph problem at merges. Addition requires shape equality; concatenation changes downstream channel offsets; normalization and projections carry the same channel semantics. A local low-importance decision therefore expands into a coupled pruning group.


## 0. Predict before running

1. Predict the exception produced by pruning only one additive branch.
2. Enumerate the tensors coupled to one output-channel deletion.
3. Explain how concat propagation differs from addition.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

A two-branch residual block with Conv-BN paths, an addition, and a consumer convolution is used. The lab records the failure from pruning one branch alone, then constructs a synchronized narrower group.

- Merge semantics determine dependency rules.
- A root channel decision propagates through producers, normalization, and consumers.
- A valid group must be checked for shape and over-pruning before mutation.


## 2. Derive the mechanism

For `z = f(x) + g(x)`, both branch outputs must have identical shapes. Removing output indices I from f requires a compatible transformation in g and changes the consumer's input dimension. With concatenation, the retained index mapping is an offset union rather than equality. Dependency graphs encode these propagation rules so one root operation yields a complete group and can be rejected before it removes every channel.

Keep value sparsity, physical shape, representation, and runtime evidence separate.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 9
LESSON_TITLE = 'Residual, Concat, and Dependency-Graph Pruning'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260817
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | an invalid one-branch channel deletion caught as a diagnostic |
| Candidate | a coupled deletion across both branches, BatchNorm state, and the consumer |
| Held constant | source block, retained indices, input, eval mode, dtype, and copied parameters |
| Measurements | captured mismatch, synchronized output shape, output drift, parameters, and latency |
| Evidence | `pytorch-gpu` |

**Experiment:** Trigger and capture an unsynchronized residual-shape failure, then build a synchronized narrow residual block.


## 5. Read the experiment code

The invalid path is wrapped in a try/except so the notebook remains successfully executed while preserving the error message as evidence. The valid path rebuilds both branches with the same retained indices and slices the consumer input channels. This is a manual miniature of a dependency group.

Do not execute until the code implements the frozen table above.


In [2]:
class Residual(nn.Module):
    def __init__(self, width=16):
        super().__init__(); self.a=nn.Conv2d(8,width,1,bias=False); self.abn=nn.BatchNorm2d(width); self.b=nn.Conv2d(8,width,3,padding=1,bias=False); self.bbn=nn.BatchNorm2d(width); self.out=nn.Conv2d(width,12,1,bias=False)
    def forward(self,x): return self.out(F.relu(self.abn(self.a(x))+self.bbn(self.b(x))))

full=Residual().to(DEVICE).eval(); x=torch.randn(4,8,20,20,device=DEVICE); keep=torch.arange(0,16,2,device=DEVICE)
bad_a=nn.Conv2d(8,8,1,bias=False,device=DEVICE)
with torch.no_grad(): bad_a.weight.copy_(full.a.weight[keep])
mismatch_captured=False; mismatch_message=""
try:
    _ = bad_a(x) + full.bbn(full.b(x))
except RuntimeError as exc:
    mismatch_captured=True; mismatch_message=str(exc).splitlines()[0]

control=copy.deepcopy(full)
remove=torch.tensor([i for i in range(16) if i not in keep.tolist()],device=DEVICE)
with torch.no_grad():
    control.a.weight[remove]=0; control.b.weight[remove]=0; control.abn.weight[remove]=0; control.abn.bias[remove]=0; control.bbn.weight[remove]=0; control.bbn.bias[remove]=0
valid=Residual(8).to(DEVICE).eval()
with torch.no_grad():
    valid.a.weight.copy_(full.a.weight[keep]); valid.b.weight.copy_(full.b.weight[keep]); valid.out.weight.copy_(full.out.weight[:,keep])
    for src,dst in ((full.abn,valid.abn),(full.bbn,valid.bbn)):
        dst.weight.copy_(src.weight[keep]); dst.bias.copy_(src.bias[keep]); dst.running_mean.copy_(src.running_mean[keep]); dst.running_var.copy_(src.running_var[keep])
with torch.inference_mode(): yc,yv=control(x),valid(x)
metrics={
    "mismatch_captured":mismatch_captured,"mismatch_message":mismatch_message,"retained_channels":int(keep.numel()),
    "valid_output_channels":int(yv.shape[1]),"valid_max_error":float((yc-yv).abs().max().item()),
    "full_parameters":count_params(full),"narrow_parameters":count_params(valid),
}
analysis=(
    f"Pruning only one additive branch produced a captured shape failure: `{mismatch_message}`. The synchronized group "
    f"retained {keep.numel()} channels across both branches, normalization state, and the consumer; it produced "
    f"{metrics['valid_output_channels']} output channels with {metrics['valid_max_error']:.3e} control drift."
)


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Mismatch captured | yes |
| Retained channels | 8 |
| Valid output channels | 12 |
| Valid max error | 0.000460 |
| Full parameters | 1,536 |
| Narrow parameters | 768 |


## 7. Interpret rather than merely print

Pruning only one additive branch produced a captured shape failure: `The size of tensor a (8) must match the size of tensor b (16) at non-singleton dimension 1`. The synchronized group retained 8 channels across both branches, normalization state, and the consumer; it produced 12 output channels with 4.603e-04 control drift.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The tensors and operators executed on CUDA through PyTorch. Native sparse-kernel identity is not inferred unless a trace or backend artifact names it.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 9,
    "title": 'Residual, Concat, and Dependency-Graph Pruning',
    "environment": ENV,
    "evidence_label": 'pytorch-gpu',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'Structural pruning at graph merges is a coupled group operation, never an isolated tensor slice.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 9,
  "title": "Residual, Concat, and Dependency-Graph Pruning",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260817
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "mismatch_captured": true,
    "mismatch_message": "The size of tensor a (8) must match the size of tensor b (16) at non-singleton dimension 1",
    "retained_channels": 8,
    "valid_output_channels": 12,
    "valid_max_error": 0.0004602670669555664,
    "full_parameters": 1536,
    "narrow_parameters": 768
  },
  "analysis": "Pruning only one additive branch produced a captured shape failure: `The size of tensor a (8) must match the size of tensor b (16) at non-singleton dimension 1`. The synchronized group retained 8 channels across both branches, normalization state, and the consumer; it produced 12 output channels with 4.603e-04 control drift.",
  "conclusion": 

## 9. Make the bounded decision

> Structural pruning at graph merges is a coupled group operation, never an isolated tensor slice.

**Acceptance/rollback:** Accept a structural mutation only when a graph-level forward check, group-size guard, and downstream shape audit pass.

**Failure analysis:** Matching shapes does not prove semantic correctness: different branches may require coordinated importance scores, grouped-convolution divisibility, or static attribute updates. Dynamic control flow can also escape a trace-based dependency graph.


## 10. Extend the evidence

Install Torch-Pruning, print the group details for an equivalent block, compare them with the manual ledger, and add a concat branch to test offset mappings.

The full evidence boundary and references are in [`README.md`](README.md).
